# Cloud Commander Simulation And Evaluation
Run XMage game simulation and checkpoint evaluation on Google's Colab CPU. GPU is not needed for the Java engine. This notebook is a Linux port that still needs a real Colab preflight; do not interpret setup as verified cloud gameplay yet.

Upload the new `cedhai-cloud.zip` bundle and your Colab `latest.pt`, then run cells in order. Checkpoints and trajectories are persistent in Drive. This is interactive notebook use, not a remote service or distributed worker. Only use trusted code/model files.


In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
bundles = [name for name in uploaded if Path(name).suffix == '.zip']
checkpoints = [name for name in uploaded if Path(name).suffix == '.pt']
if len(bundles) != 1 or len(checkpoints) != 1 or len(uploaded) != 2:
    raise ValueError('Upload exactly cedhai-cloud.zip and latest.pt')
archive_path = bundles[0]
checkpoint_upload = checkpoints[0]


In [ ]:
import hashlib, json, zipfile
from pathlib import Path, PurePosixPath
workspace = Path('/content/cedhai-cloud')
workspace.mkdir(exist_ok=False)
with zipfile.ZipFile(archive_path) as archive:
    names = archive.namelist()
    metadata = json.loads(archive.read('bundle.json'))
    if len(names) != len(set(names)) or metadata['schema_version'] != 1 or set(names) != set(metadata['sha256']) | {'bundle.json'}:
        raise ValueError('Invalid bundle manifest')
    for name, expected in metadata['sha256'].items():
        parts = PurePosixPath(name).parts
        if not parts or parts[0] not in ('project', 'dataset') or '..' in parts or '\\' in name:
            raise ValueError('Unsafe archive path')
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError('Bundle checksum mismatch')
        target = workspace.joinpath(*parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
print('Source revision:', metadata['git_revision'])


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(workspace / 'project') + '[training]'], check=True)
subprocess.run([sys.executable, '-m', 'cedhai.cloud_setup'], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
RUN_NAME = 'cloud-evaluation-v1'
run_root = Path('/content/drive/MyDrive/cedhai/cloud') / RUN_NAME
run_root.mkdir(parents=True, exist_ok=True)
for source, name in ((Path(checkpoint_upload), 'latest.pt'), (Path(archive_path), 'source-bundle.zip')):
    destination = run_root / name
    if destination.exists():
        if hashlib.sha256(destination.read_bytes()).digest() != hashlib.sha256(source.read_bytes()).digest():
            raise ValueError('Saved inputs differ: use the same files to resume, or change RUN_NAME')
    else:
        shutil.copyfile(source, destination)
checkpoint_path = run_root / 'latest.pt'
print('Persistent run:', run_root)


## Real Linux Preflight
This four-turn game tests the downloaded rules engine, Java compiler, socket protocol, deck fixtures, and checkpoint agent. It deliberately truncates and does not provide a win/loss result. If it fails, keep the logs in Drive and stop before the batch.


In [ ]:
preflight = run_root / 'preflight'
attempt = 1
while preflight.exists():
    result = preflight / 'result.json'
    if result.exists() and json.loads(result.read_text()).get('success'):
        break
    attempt += 1
    preflight = run_root / f'preflight-attempt{attempt}'
if not (preflight / 'result.json').exists():
    subprocess.run([sys.executable, '-m', 'cedhai.play_model', '--checkpoint', str(checkpoint_path),
                    '--out', str(preflight), '--seed', '20', '--turns', '4'], check=True)
print('Preflight:', preflight)


## Seating-Balanced Checkpoint Evaluation
Twenty sequential games: five fresh seeds, four seating rotations each. The model faces three scripted blocking controls. A session interruption preserves completed games; reconnect, upload the SAME files, use the SAME RUN_NAME, and rerun. Resume rejects changed inputs/configuration, skips completed games, and preserves partial attempts.

Seeds 7, 10, 11, 12, 13, and 20 are development/preflight seeds, excluded here. Do not tune on this batch and subsequently call it untouched test data.


In [ ]:
evaluation = run_root / 'evaluation'
command = [sys.executable, '-m', 'cedhai.evaluate', '--checkpoint', str(checkpoint_path),
           '--policies', 'model', 'defend', 'defend', 'defend',
           '--seeds', '14', '16', '17', '18', '19',
           '--development-seeds', '7', '10', '11', '12', '13', '20',
           '--out', str(evaluation)]
if (evaluation / 'evaluation.json').exists():
    command.append('--resume')
subprocess.run(command, check=True)


In [ ]:
evaluation_report = json.loads((evaluation / 'evaluation.json').read_text())
print(json.dumps(evaluation_report['summary'], indent=2))
diagnostics_path = run_root / 'diagnostics.json'
if not diagnostics_path.exists():
    subprocess.run([sys.executable, '-m', 'cedhai.policy_diagnostics', '--evaluation',
                    str(evaluation / 'evaluation.json'), '--out', str(diagnostics_path)], check=True)
print(json.dumps(json.loads(diagnostics_path.read_text()), indent=2))


In [ ]:
import zipfile
summary_zip = run_root / 'evaluation-summary.zip'
with zipfile.ZipFile(summary_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(evaluation / 'evaluation.json', 'evaluation.json')
    archive.write(diagnostics_path, 'diagnostics.json')
files.download(str(summary_zip))
print('Full trajectories/logs remain in Drive:', evaluation)


## Optional: Generate A Larger Training Dataset In Colab
This separate batch uses seeds 21-26 and scripted/random controls, not the held-out evaluation seeds. It creates twenty-four games. Seed 25 is validation under the current split rule. Training remains an imitation smoke experiment, not self-play RL. These optional cells may consume substantial CPU time and Drive space.


In [ ]:
collection = run_root / 'collection'
command = [sys.executable, '-m', 'cedhai.evaluate', '--seeds', '21', '22', '23', '24', '25', '26', '--out', str(collection)]
if (collection / 'evaluation.json').exists():
    command.append('--resume')
subprocess.run(command, check=True)


In [ ]:
collection_report = json.loads((collection / 'evaluation.json').read_text())
dataset_path = run_root / 'expanded-dataset'
if not dataset_path.exists():
    sources = [game['run'] for game in collection_report['games'] if game['success']]
    subprocess.run([sys.executable, '-m', 'cedhai.learning_data', '--runs', *sources,
                    '--out', str(dataset_path)], check=True)
subprocess.run([sys.executable, '-m', 'cedhai.dataset_report', '--dataset', str(dataset_path)], check=True)


In [ ]:
expanded_model = run_root / 'expanded-model'
command = [sys.executable, '-m', 'cedhai.train', '--dataset', str(dataset_path), '--out', str(expanded_model),
           '--epochs', '20', '--device', 'cpu']
subprocess.run(command, check=True)
files.download(str(expanded_model / 'latest.pt'))
print('Do not repeatedly optimize on the held-out evaluation seeds; use a new evaluation set for the next model.')
